In [2]:
import numpy as np
import pandas as pd
import re

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

In [5]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("XGBoost_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

In [6]:
import json
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

/home/elianaostro/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
param_list = {}

In [8]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [9]:
df = pd.read_csv('data/train/cleaned_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Año                      14603 non-null  float64
 4   Versión                  14085 non-null  object 
 5   Color                    14288 non-null  object 
 6   Tipo de combustible      14603 non-null  object 
 7   Puertas                  14603 non-null  float64
 8   Transmisión              14590 non-null  object 
 9   Motor                    14410 non-null  float64
 10  Tipo de carrocería       14603 non-null  object 
 11  Con cámara de retroceso  3749 non-null   object 
 12  Kilómetros               14603 non-null  float64
 13  Título                   14603 non-null  object 
 14  Precio                

In [10]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 39)
Val: (2921, 39)


## First Experiment

Ajuste de hiperparametros

In [11]:
# Espacio de hiperparámetros refinado
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

# Modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"]=best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.36e+13   |   3.92e+13
R²           |   0.9730     |   0.9133
MAE          |   1.79e+06   |   2.73e+06
EVS          |   0.9730     |   0.9134
Max Error    |   2.11e+08   |   2.01e+08
Median AE    |   1.15e+06   |   1.45e+06

Best parameters found:
{
    "subsample": 1.0,
    "reg_lambda": 1,
    "reg_alpha": 0.01,
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "gamma": 0,
    "colsample_bytree": 0.6
}


In [12]:
# Espacio de hiperparámetros refinado
param_dist = {
    'learning_rate': [0.05, 0.075, 0.1, 0.125],
    'n_estimators': [250, 300, 350, 400],
    'max_depth': [5, 6, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.005, 0.01, 0.05],
    'reg_lambda': [0.8, 1, 1.2, 1.5],
    'min_child_weight': [1, 3, 5]
}

# Modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["RandomizedSearchCV_2"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.44e+13   |   4.60e+13
R²           |   0.9715     |   0.8982
MAE          |   1.90e+06   |   2.80e+06
EVS          |   0.9715     |   0.8983
Max Error    |   2.09e+08   |   2.34e+08
Median AE    |   1.24e+06   |   1.51e+06

Best parameters found:
{
    "subsample": 0.9,
    "reg_lambda": 1.2,
    "reg_alpha": 0.05,
    "n_estimators": 350,
    "min_child_weight": 1,
    "max_depth": 5,
    "learning_rate": 0.125,
    "gamma": 0.1,
    "colsample_bytree": 0.7
}


In [13]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    model = XGBRegressor(**params, objective='reg:squarederror', random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=50)
best_params = study.best_params

# entrenar el modelo con los mejores hiperparámetros
best_model = XGBRegressor(**best_params, objective='reg:squarederror', random_state=42)
best_model.fit(X_train, y_train)
# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)
# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["Optuna"] = best_params
print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   9.20e+13   |   1.06e+14
R²           |   0.8182     |   0.7662
MAE          |   5.48e+06   |   5.57e+06
EVS          |   0.8182     |   0.7664
Max Error    |   3.10e+08   |   2.92e+08
Median AE    |   4.07e+06   |   4.03e+06

Best parameters found:
{
    "max_depth": 5,
    "learning_rate": 0.01003021809013797,
    "n_estimators": 121,
    "subsample": 0.5030677567434054,
    "colsample_bytree": 0.9451030475928729,
    "reg_alpha": 0.24965418915405826,
    "reg_lambda": 0.0009534928487342291
}


In [ ]:
halving_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
halving_search.fit(X_train, y_train)
best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)
# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["HalvingRandomSearchCV"] = best_params


In [ ]:
input_example = X_train.iloc[0:1]
artifact_csv_path = "data/train/cleaned_train.csv"  # ajusta la ruta si es necesario

for name, params in param_list.items():
    model = XGBRegressor(objective='reg:squarederror', random_state=42, **params)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"XGBoost_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.xgboost.log_model(model, artifact_path="model", input_example=input_example, signature=signature)

2025/06/17 12:36:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[12:36:50] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_RandomizedSearchCV at: http://localhost:5000/#/experiments/289899895095468934/runs/ec31d036609b43558db8e7d8679f8e06
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934


2025/06/17 12:36:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[12:36:53] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_RandomizedSearchCV_2 at: http://localhost:5000/#/experiments/289899895095468934/runs/0ad5abb3781a4bae9ac7938f6d255464
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934


2025/06/17 12:36:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[12:36:56] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_Optuna at: http://localhost:5000/#/experiments/289899895095468934/runs/b211af23eb6e4b6fa1c56cd25fe6c279
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934


2025/06/17 12:36:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[12:36:59] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_HalvingRandomSearchCV at: http://localhost:5000/#/experiments/289899895095468934/runs/2ad6c7b8941c474a9fcaa5eedb5c0800
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934
